In [ ]:
import pandas as pd

from pathlib import Path


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir():
            return p
    raise FileNotFoundError("Could not find project root containing partition/.")


BASE = str(find_project_root())

df = pd.read_excel(BASE + r"\partition\Validation\Val1\Train.xlsx")

print(df.shape)
print(df.columns.tolist())
print(df.head())

In [ ]:
df['patient_id'] = df['image_name'].str.split('_Block').str[0]

print("Pacients únics al train Val1:", df['patient_id'].nunique())
print(df['patient_id'].value_counts().head(10))

In [ ]:
# Convert one-hot encoding to a single label
df['label'] = df[['NC', 'G3', 'G4', 'G5']].idxmax(axis=1)

print("Class distribution in Val1 train:")
print(df['label'].value_counts())
print()
print("En percentatge:")
print(df['label'].value_counts(normalize=True).round(3) * 100)

In [ ]:
resultats = []

for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    for split in ['Train', 'Test']:
        path = BASE + rf"\partition\Validation\{fold}\{split}.xlsx"
        df = pd.read_excel(path)
        df['label'] = df[['NC', 'G3', 'G4', 'G5']].idxmax(axis=1)
        df['patient_id'] = df['image_name'].str.split('_Block').str[0]
        
        counts = df['label'].value_counts()
        resultats.append({
            'fold': fold,
            'split': split,
            'n_patches': len(df),
            'n_patients': df['patient_id'].nunique(),
            'NC%': round(counts.get('NC', 0) / len(df) * 100, 1),
            'G3%': round(counts.get('G3', 0) / len(df) * 100, 1),
            'G4%': round(counts.get('G4', 0) / len(df) * 100, 1),
            'G5%': round(counts.get('G5', 0) / len(df) * 100, 1),
        })

resum = pd.DataFrame(resultats)
print(resum.to_string(index=False))

In [ ]:
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    train = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Train.xlsx")
    test = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Test.xlsx")
    
    train_patients = set(train['image_name'].str.split('_Block').str[0])
    test_patients = set(test['image_name'].str.split('_Block').str[0])
    
    overlap = train_patients & test_patients
  print(f"{fold}: {len(overlap)} patients overlapping")

In [ ]:
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    train = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Train.xlsx")
    test = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Test.xlsx")
    
    train_patients = set(train['image_name'].str.split('_Block').str[0])
    test_patients = set(test['image_name'].str.split('_Block').str[0])
    
  print(f"{fold}: train={len(train_patients)} patients, test={len(test_patients)} patients, total={len(train_patients | test_patients)}")

In [ ]:
official_test = pd.read_excel(BASE + r"\partition\Test\Test.xlsx")
train_official = pd.read_excel(BASE + r"\partition\Test\Train.xlsx")

test_patients = set(official_test['image_name'].str.split('_Block').str[0])
train_patients = set(train_official['image_name'].str.split('_Block').str[0])

print(f"Official test: {len(test_patients)} patients")
print(f"Official train: {len(train_patients)} patients")
print(f"Total: {len(test_patients | train_patients)} patients")

In [ ]:
val_patients = set()
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    df_fold = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Train.xlsx")
    val_patients |= set(df_fold['image_name'].str.split('_Block').str[0])

official_test_patients = set(official_test['image_name'].str.split('_Block').str[0])

overlap = val_patients & official_test_patients
print(f"Overlapping patients between folds and official test: {len(overlap)}")

In [ ]:
df

In [ ]:
image_name

In [ ]:
mask

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Take the name of the first image of the Val1 Train
image_name = df['image_name'].iloc[0]

img  = mpimg.imread(BASE + r"\images\\" + image_name)
mask = mpimg.imread(BASE + r"\masks\\"  + image_name )

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img)
axes[0].set_title('Original image')
axes[0].axis('off')
axes[1].imshow(mask)
axes[1].set_title('Gleason mask')
axes[1].axis('off')

plt.suptitle(image_name, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Define colors by class: NC=green, G3=yellow, G4=orange, G5=red
cmap = {0: [0, 255, 0], 1: [255, 255, 0], 2: [255, 165, 0], 3: [255, 0, 0]}

mask_raw = mpimg.imread(BASE + r"\masks\\" + image_name)
print("Unique values in the mask:", np.unique(mask_raw))
print("Shape:", mask_raw.shape)

In [ ]:
mask_rgb = mpimg.imread(BASE + r"\masks\\" + image_name)
print("Shape:", mask_rgb.shape)
print("Dtype:", mask_rgb.dtype)

# Inspect the first pixels
print("\nFirst pixels:")
print(mask_rgb[:3, :3])

In [ ]:
import cv2

mask_cv = cv2.imread(BASE + r"\masks\\" + image_name, cv2.IMREAD_UNCHANGED)
print("Shape:", mask_cv.shape)
print("Dtype:", mask_cv.dtype)
print("Unique values:", np.unique(mask_cv))
print("\nFirst pixels:")
print(mask_cv[:3, :3])

In [ ]:
import os

mask_path = BASE + r"\masks\\" + image_name
print("Path:", mask_path)
print("Exists?", os.path.exists(mask_path))

# List the first files in the masks folder
print("\nFirst files in masks:")
print(os.listdir(BASE + r"\masks")[:5])

In [ ]:
from PIL import Image

mask_pil = Image.open(mask_path)
print("Mode:", mask_pil.mode)
print("Size:", mask_pil.size)

mask_arr = np.array(mask_pil)
print("Shape:", mask_arr.shape)
print("Dtype:", mask_arr.dtype)
print("Unique values:", np.unique(mask_arr))
print("\nFirst pixels:")
print(mask_arr[:3, :3])

In [ ]:
import os
import numpy as np
from PIL import Image

from pathlib import Path


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir():
            return p
    raise FileNotFoundError("Could not find project root containing partition/.")


BASE = str(find_project_root())
mask_dir = os.path.join(BASE, "masks")

valors = set()
for fname in os.listdir(mask_dir):
    if not fname.lower().endswith((".png", ".tif", ".jpg")):
        continue
    arr = np.array(Image.open(os.path.join(mask_dir, fname)))
    valors.update(np.unique(arr).tolist())
    
print("unique values found across all masks:", sorted(valors))

In [ ]:
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    train = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Train.xlsx")
    test = pd.read_excel(BASE + rf"\partition\Validation\{fold}\Test.xlsx")
    
    train_p = train['image_name'].str.split('_Block').str[0].nunique()
    test_p = test['image_name'].str.split('_Block').str[0].nunique()
    
  print(f"{fold}: {train_p} train + {test_p} test = {train_p + test_p} patients")

In [ ]:
wsi = pd.read_excel(BASE + r"\wsi_labels.xlsx")
print(wsi.columns.tolist())
print(wsi.head(10))

In [ ]:
biopsies_per_patient = wsi.groupby('patient_id')['slide_id'].count().sort_values(ascending=False)
print("Biopsy distribution per patient:")
print(biopsies_per_patient.value_counts().sort_index())
print("\nPatients with most biopsies:")
print(biopsies_per_patient.head(10))

In [ ]:
dfs = []

# Cross-validation folds
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    for split in ['Train', 'Test']:
        df_tmp = pd.read_excel(BASE + rf"\partition\Validation\{fold}\{split}.xlsx")
        df_tmp['fold'] = fold
        df_tmp['split'] = split
        dfs.append(df_tmp)

# Official test
for split in ['Train', 'Test']:
    df_tmp = pd.read_excel(BASE + rf"\partition\Test\{split}.xlsx")
    df_tmp['fold'] = 'Official_Test'
    df_tmp['split'] = split
    dfs.append(df_tmp)

# Merge everything
df_all = pd.concat(dfs, ignore_index=True)

# Add useful columns
df_all['patient_id'] = df_all['image_name'].str.split('_Block').str[0]
df_all['label'] = df_all[['NC', 'G3', 'G4', 'G5']].idxmax(axis=1)

print("Shape total:", df_all.shape)
print("Columns:", df_all.columns.tolist())
print(df_all.head(3))

In [ ]:
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    df_fold = df_all[df_all['fold'] == fold]
    train = df_fold[df_fold['split'] == 'Train']
    test = df_fold[df_fold['split'] == 'Test']
    
  print(f"\n--- {fold} ---")
  print(f"Train: {train['patient_id'].nunique()} patients, {len(train)} patches")
  print(f"Test: {test['patient_id'].nunique()} patients, {len(test)} patches")
  print(f"Train classes: NC={round(train['NC'].sum()/len(train)*100,1)}% G3={round(train['G3'].sum()/len(train)*100,1)}% G4={round(train['G4'].sum()/len(train)*100,1)}% G5={round(train['G5'].sum()/len(train)*100,1)}%")
  print(f"Test classes: NC={round(test['NC'].sum()/len(test)*100,1)}% G3={round(test['G3'].sum()/len(test)*100,1)}% G4={round(test['G4'].sum()/len(test)*100,1)}% G5={round(test['G5'].sum()/len(test)*100,1)}%")

In [ ]:
print(wsi.columns.tolist())
print(wsi.head(10))

In [ ]:
import os
import pandas as pd

from pathlib import Path


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir():
            return p
    raise FileNotFoundError("Could not find project root containing partition/.")


BASE = str(find_project_root())

# load slide→patient mapping
wsi = pd.read_excel(os.path.join(BASE, "wsi_labels.xlsx"))

# build global dataframe as before
dfs = []
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    for split in ['Train', 'Test']:
        df_tmp = pd.read_excel(os.path.join(BASE, "partition", "Validation", fold, f"{split}.xlsx"))
        df_tmp['fold'] = fold
        df_tmp['split'] = split
        dfs.append(df_tmp)
for split in ['Train', 'Test']:
    df_tmp = pd.read_excel(os.path.join(BASE, "partition", "Test", f"{split}.xlsx"))
    df_tmp['fold'] = 'Official_Test'
    df_tmp['split'] = split
    dfs.append(df_tmp)

df_all = pd.concat(dfs, ignore_index=True)

# extract slide_id and merge with wsi_labels to get the true patient_id
df_all['slide_id'] = df_all['image_name'].str.split('_Block').str[0]
df_all = df_all.merge(wsi[['slide_id', 'patient_id']], on='slide_id', how='left')

# comprovar que no hi hagi valors perduts
if df_all['patient_id'].isna().any():
  print("!! imatges sense pacient in wsi_labels:", df_all.loc[df_all['patient_id'].isna(), 'slide_id'].unique())

# etiqueta única
df_all['label'] = df_all[['NC', 'G3', 'G4', 'G5']].idxmax(axis=1)

# nombre de patients reals
unique_patients = df_all['patient_id'].nunique()
print("Pacients únics in tot el dataset:", unique_patients)
assert unique_patients == 95, "no coincideixen els 95 patients esperats"

# resum per fold/split
for fold in ['Val1', 'Val2', 'Val3', 'Val4']:
    df_fold = df_all[df_all['fold'] == fold]
    train = df_fold[df_fold['split'] == 'Train']
    test  = df_fold[df_fold['split'] == 'Test']

    pts_train = set(train['patient_id'])
    pts_test  = set(test['patient_id'])
    pts_fold  = pts_train | pts_test

  print(f"\n--- {fold} ---")
  print(f"patients: train={len(pts_train)} test={len(pts_test)} total={len(pts_fold)}")
  print(f"patches: train={len(train):6d} test={len(test):6d}")
    for cl in ['NC','G3','G4','G5']:
        pct_train = round(train[cl].sum()/len(train)*100,1)
        pct_test  = round(test[cl].sum()/len(test)*100,1)
    print(f"  {cl}: train={pct_train:4.1f}% test={pct_test:4.1f}%")

# optional: taula resumida
summary = []
for fold in ['Val1','Val2','Val3','Val4']:
    df_fold = df_all[df_all['fold'] == fold]
    for split in ['Train','Test']:
        df_s = df_fold[df_fold['split'] == split]
        pts = set(df_s['patient_id'])
        summary.append({
            'fold': fold, 'split': split,
            'n_patches': len(df_s),
            'n_patients': len(pts),
            'NC%': round(df_s['NC'].sum()/len(df_s)*100,1),
            'G3%': round(df_s['G3'].sum()/len(df_s)*100,1),
            'G4%': round(df_s['G4'].sum()/len(df_s)*100,1),
            'G5%': round(df_s['G5'].sum()/len(df_s)*100,1),
        })
summary_df = pd.DataFrame(summary)
print("\nResum in taula:")
print(summary_df)

# verificació extra: la unió dels quatre folds
unio_folds = set()
for fold in ['Val1','Val2','Val3','Val4']:
    df_fold = df_all[df_all['fold']==fold]
    unio_folds |= set(df_fold['patient_id'])
print("\nPacients únics in els quatre folds:", len(unio_folds)) # 95 també

In [ ]:
test_of = df_all[df_all['fold'] == 'Official_Test']
train_of = test_of[test_of['split'] == 'Train']
test_of_test = test_of[test_of['split'] == 'Test']

print(f"Official test train: {train_of['patient_id'].nunique()} patients, {train_of['slide_id'].nunique()} WSIs, {len(train_of)} patches")
print(f"Official test test: {test_of_test['patient_id'].nunique()} patients, {test_of_test['slide_id'].nunique()} WSIs, {len(test_of_test)} patches")

In [ ]:
df_all

In [ ]:
print("Total files df_all:", len(df_all))
print("Unique patches:", df_all['image_name'].nunique())
print("Mitjana d'aparicions per patch:", round(len(df_all) / df_all['image_name'].nunique(), 1))

In [ ]:
print(f"Total number of images: {df_all['image_name'].nunique()}")

In [ ]:
import os

images_dir = os.path.join(BASE, "images")
num_images = len(os.listdir(images_dir))
print(f"Total number of images in the folder: {num_images}")

In [ ]:
import os

# Get images from the folder
images_dir = os.path.join(BASE, "images")
images_folder = set(os.listdir(images_dir))

# Get images referenced in df_all
images_df = set(df_all['image_name'].unique())

# Images in the folder but NOT in df_all
imatges_no_referenced = images_folder - images_df
print(f"Images in the folder but NOT in df_all: {len(imatges_no_referenced)}")

# Images in df_all but NOT in the folder (this should not happen)
imatges_missing = images_df - images_folder
print(f"Images in df_all but NOT in the folder: {len(imatges_missing)}")

# Show first examples of missing images
if imatges_no_referenced:
  print(f"\nExamples of non-referenced images:")
    for img in list(imatges_no_referenced)[:10]:
    print(f" {img}")

In [ ]:
# All images available on disk
imatges_disc = set([f for f in os.listdir(BASE + r"\images")])

# All images that appear in folds
imatges_folds = set(df_all['image_name'].unique())

# Images on disk but not in folds
nomes_disc = imatges_disc - imatges_folds

# Images in folds but not on disk (should not happen)
nomes_folds = imatges_folds - imatges_disc

print(f"Images on disk:    {len(imatges_disc)}")
print(f"Images in folds:   {len(imatges_folds)}")
print(f"Only on disk:     {len(nomes_disc)}")
print(f"Only in folds:    {len(nomes_folds)}")

# Which patients own the missing images?
nomes_disc_df = pd.DataFrame({'image_name': list(nomes_disc)})
nomes_disc_df['slide_id'] = nomes_disc_df['image_name'].str.split('_Block').str[0]
print(f"\nWSIs with images not assigned to any fold: {nomes_disc_df['slide_id'].nunique()}")

In [ ]:
# WSIs that appear in folds
wsis_folds = set(df_all['slide_id'].unique())

# WSIs from unassigned images
wsis_nomes_disc = set(nomes_disc_df['slide_id'].unique())

# How many of these WSIs appear in folds?
wsis_compartides = wsis_folds & wsis_nomes_disc
wsis_noves = wsis_nomes_disc - wsis_folds

print(f"Unassigned WSIs that DO appear in folds: {len(wsis_compartides)}")
print(f"Unassigned WSIs that DO NOT appear in folds: {len(wsis_noves)}")
print(f"\nExample new WSIs (not in folds):")
print(list(wsis_noves)[:10])

In [ ]:
# Patches per WSI on disk
patches_disc = nomes_disc_df.groupby('slide_id').size().reset_index(name='patches_disc')

# Patches per WSI in folds (unique only)
patches_folds = df_all.drop_duplicates('image_name').groupby('slide_id').size().reset_index(name='patches_folds')

# Merge
comparacio = patches_disc.merge(patches_folds, on='slide_id', how='outer').fillna(0)
comparacio['total'] = comparacio['patches_disc'] + comparacio['patches_folds']
comparacio = comparacio.sort_values('patches_disc', ascending=False)

print(comparacio.head(15))
print(f"\nTotal unassigned patches: {int(comparacio['patches_disc'].sum())}")
print(f"Total assigned patches:  {int(comparacio['patches_folds'].sum())}")

In [ ]:
# Load labels for assigned images
labels_assigned = df_all.drop_duplicates('image_name')[['image_name', 'label']]

# For unassigned images, labels are not available in the Excel files...
# But we can check whether there is a pattern in WSIs with more unassigned patches
comparacio_wsi = comparacio.merge(
    wsi[['slide_id', 'patient_id', 'Gleason_primary', 'Gleason_secondary']], 
    on='slide_id', how='left'
)

print("Primary Gleason for WSIs with more unassigned patches:")
print(comparacio_wsi[['slide_id', 'patches_disc', 'patches_folds', 'Gleason_primary']].head(15))
print("\nPrimary Gleason distribution (WSIs with unassigned patches):")
print(comparacio_wsi['Gleason_primary'].value_counts())

In [ ]:
# Benign WSIs (Gleason_primary = 0)
wsis_benign = wsi[wsi['Gleason_primary'] == 0]['slide_id'].tolist()

print(f"Total Benign WSIs: {len(wsis_benign)}")

# How many of them have patches in folds?
wsis_benign_folds = df_all[df_all['slide_id'].isin(wsis_benign)]['slide_id'].nunique()
wsis_benign_disc = nomes_disc_df[nomes_disc_df['slide_id'].isin(wsis_benign)]['slide_id'].nunique()

print(f"Benign WSIs with patches in folds: {wsis_benign_folds}")
print(f"Benign WSIs with patches ONLY on disk: {wsis_benign_disc}")

In [ ]:
# Patches on disk for all WSIs
patches_disk_all = pd.DataFrame({
    'image_name': list(imatges_disc)
})
patches_disk_all['slide_id'] = patches_disk_all['image_name'].str.split('_Block').str[0]
patches_disk_all = patches_disk_all.groupby('slide_id').size().reset_index(name='patches_disc')

# Patches in folds for all WSIs
patches_folds_all = df_all.drop_duplicates('image_name').groupby('slide_id').size().reset_index(name='patches_folds')

# Merge with wsi_labels
comparison_full = patches_disk_all.merge(patches_folds_all, on='slide_id', how='outer').fillna(0)
comparison_full = comparison_full.merge(wsi[['slide_id', 'Gleason_primary']], on='slide_id', how='left')
comparison_full['type'] = comparison_full['Gleason_primary'].apply(
    lambda x: 'Benign (NC)' if x == 0 else 'Cancer'
)

summary_full = comparison_full.groupby('type').agg(
    n_wsis=('slide_id', 'count'),
    patches_disc=('patches_disc', 'sum'),
    patches_folds=('patches_folds', 'sum'),
)
summary_full['total_patches'] = summary_full['patches_disc'] + summary_full['patches_folds']
summary_full['pct_selected'] = round(summary_full['patches_folds'] / summary_full['total_patches'] * 100, 1)

print(summary_full)
print(f"\nTotal patches on disk:  {int(summary_full['total_patches'].sum())}")
print(f"Total patches in folds: {int(summary_full['patches_folds'].sum())}")

In [ ]:
# patches_disc already includes ALL images on disk (assigned and unassigned)
# patches_folds is a subset of patches_disc

summary_consistent = comparison_full.groupby('type').agg(
    n_wsis=('slide_id', 'count'),
    total_disc=('patches_disc', 'sum'),
    in_folds=('patches_folds', 'sum'),
)

summary_consistent['unassigned'] = summary_consistent['total_disc'] - summary_consistent['in_folds']
summary_consistent['pct_selected'] = round(summary_consistent['in_folds'] / summary_consistent['total_disc'] * 100, 1)

print(summary_consistent)
print(f"\nTotal patches on disk:    {int(summary_consistent['total_disc'].sum())}")
print(f"Total patches in folds:   {int(summary_consistent['in_folds'].sum())}")
print(f"Total unassigned patches:  {int(summary_consistent['unassigned'].sum())}")

In [ ]:
import os

# Get images from the folder
images_dir = os.path.join(BASE, "images")
images_folder = set(os.listdir(images_dir))

# Get images referenced in df_all
images_df = set(df_all['image_name'].unique())

# Images in the folder but NOT in df_all
imatges_no_referenced = images_folder - images_df

print(f"Total non-referenced images: {len(imatges_no_referenced)}")

# Check whether any are in the official test
official_test = df_all[df_all['fold'] == 'Official_Test']
images_official_test = set(official_test['image_name'].unique())

images_not_ref_in_test = imatges_no_referenced & images_official_test
images_not_ref_outside_test = imatges_no_referenced - images_official_test

print(f"Non-referenced images that ARE in the official test: {len(images_not_ref_in_test)}")
print(f"Non-referenced images that are NOT in df_all: {len(images_not_ref_outside_test)}")

# Examples of those not present
if images_not_ref_outside_test:
  print(f"\nImage examples que no estan a cap partició:")
    for img in list(images_not_ref_outside_test)[:10]:
    print(f" {img}")